# Course 6 — Comparing Classifiers: When to Use What

A head-to-head comparison of all classification methods covered in Week 4,
benchmarked on real datasets with guidance on when each method wins.

**Sections**
1. Methods at a glance — assumptions, boundary shape, parameter counts (0:00–0:15)
2. Four ISLR scenarios — which method wins and why (0:15–0:35)
3. Benchmark on Default, Iris, and simulated data (0:35–0:55)
4. Decision guide and preview of non-linear methods (0:55–1:00)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.datasets import make_classification, make_moons

MODELS = [
    ('LR',  LogisticRegression(max_iter=2000)),
    ('LDA', LinearDiscriminantAnalysis()),
    ('QDA', QuadraticDiscriminantAnalysis()),
    ('NB',  GaussianNB()),
]

default = load_dataset('default')
iris    = load_dataset('iris')
d = default.copy()
d['y'] = (d['default'] == 'Yes').astype(int)
print('Setup complete.')

## 1. Methods at a Glance

| Method | What is modelled | Boundary | Key assumption | Scales to high p? |
|---|---|---|---|---|
| Logistic Regression | P(Y=k\|X) directly | Linear | None (discriminative) | Yes (L1/L2) |
| LDA | f_k(x) = N(μ_k, Σ) | Linear | Gaussian, shared Σ | No (p > n fails) |
| QDA | f_k(x) = N(μ_k, Σ_k) | Quadratic | Gaussian, per-class Σ | No (worse than LDA) |
| Naive Bayes | f_k(x) = ∏_j f_kj(x_j) | Depends | Cond. independence | Yes (always) |

### Parameter counts for covariance alone (p features, K classes)

| Method | Covariance parameters |
|---|---|
| LDA | p(p+1)/2 (shared) |
| QDA | K · p(p+1)/2 |
| Naive Bayes | p · K (diagonal only) |

With p=100 features, K=2 classes: LDA=5050, QDA=10100, NB=200.

No single classifier dominates across all settings — the right choice depends on n, p, class distribution, and whether the true boundary is linear or curved.

In [ ]:
# Visualize how parameter counts scale with p
p_vals = np.arange(2, 201)
K = 3

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].semilogy(p_vals, p_vals*(p_vals+1)//2, label='LDA (shared Σ)', linewidth=2)
axes[0].semilogy(p_vals, K*p_vals*(p_vals+1)//2, label='QDA (per-class Σ)', linewidth=2)
axes[0].semilogy(p_vals, p_vals*K, label='Naive Bayes (diagonal)', linewidth=2)
axes[0].semilogy(p_vals, p_vals*np.ones_like(p_vals), label='LR (just β)', linewidth=2, linestyle='--')
axes[0].set_xlabel('p (number of features)'); axes[0].set_ylabel('Parameters (log scale)')
axes[0].set_title(f'Covariance parameter count (K={K} classes)')
axes[0].legend(fontsize=9)

# What happens when p approaches n?
n = 200
axes[1].axvline(n, color='k', linestyle='--', alpha=0.5, label=f'n = {n}')
axes[1].fill_between(p_vals, 0, 1, where=p_vals >= n, alpha=0.15, color='C3',
                      label='p ≥ n: LDA/QDA fail (singular Σ)')
axes[1].fill_between(p_vals, 0, 1, where=p_vals < n, alpha=0.15, color='C2',
                      label='p < n: all methods feasible')
axes[1].set_xlabel('p (number of features)'); axes[1].set_ylabel('')
axes[1].set_title('Feasibility regions (n=200)')
axes[1].set_yticks([]); axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

## 2. Four Scenarios (ISLR Section 4.5)

ISLR simulates four settings to show which classifier wins and why.
We reproduce similar experiments below.

These scenarios map directly onto the bias-variance tradeoff: when classes are truly Gaussian and Σ is shared, LDA achieves the Bayes optimal error rate with minimal variance; when the boundary is non-linear or the Gaussian assumption breaks down, the lower-bias flexibility of LR, KNN, or NB can all win depending on sample size.

In [ ]:
# Scenario 1: Gaussian classes, same Σ — LDA is Bayes optimal
rng = np.random.default_rng(0)
Sigma = np.array([[1.0, 0.5], [0.5, 1.0]])  # shared covariance
n_per_class = 100
X0 = rng.multivariate_normal([0, 0], Sigma, n_per_class)
X1 = rng.multivariate_normal([2, 2], Sigma, n_per_class)
X_s1 = np.vstack([X0, X1]); y_s1 = np.array([0]*n_per_class + [1]*n_per_class)

print('Scenario 1: Gaussian, same Σ — LDA should win')
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
# same preprocessing for all classifiers ensures fair comparison
for name, model in MODELS:
    # cross-validation gives a less noisy accuracy estimate than a single train/test split
    scores = cross_val_score(model, X_s1, y_s1, cv=cv, scoring='roc_auc')
    print(f'  {name:5s}: AUC = {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# Scenario 2: Gaussian classes, DIFFERENT Σ_k — QDA should win
Sigma0 = np.array([[1.0, 0.8], [0.8, 1.0]])
Sigma1 = np.array([[1.5, -0.7], [-0.7, 0.8]])
X0_s2 = rng.multivariate_normal([0, 0], Sigma0, n_per_class)
X1_s2 = rng.multivariate_normal([2, 1], Sigma1, n_per_class)
X_s2 = np.vstack([X0_s2, X1_s2]); y_s2 = np.array([0]*n_per_class + [1]*n_per_class)

print('Scenario 2: Gaussian, different Σ_k — QDA should win')
for name, model in MODELS:
    scores = cross_val_score(model, X_s2, y_s2, cv=cv, scoring='roc_auc')
    print(f'  {name:5s}: AUC = {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# Scenario 3: Non-Gaussian features (skewed / bimodal) — LR should win
X0_s3 = np.column_stack([rng.exponential(1, n_per_class),
                          rng.exponential(2, n_per_class)])
X1_s3 = np.column_stack([rng.exponential(2, n_per_class) + 1,
                          rng.exponential(1, n_per_class) + 2])
X_s3 = np.vstack([X0_s3, X1_s3]); y_s3 = np.array([0]*n_per_class + [1]*n_per_class)

print('Scenario 3: Non-Gaussian features (exponential) — LR should be robust')
for name, model in MODELS:
    scores = cross_val_score(model, X_s3, y_s3, cv=cv, scoring='roc_auc')
    print(f'  {name:5s}: AUC = {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# Scenario 4: High-dimensional (p >> n) — NB and LR win; LDA/QDA fail
print('Scenario 4: High-dimensional (p=500, n=100) — NB should win')
X_hi, y_hi = make_classification(n_samples=100, n_features=500, n_informative=20,
                                   n_redundant=0, random_state=0)
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(X_hi, y_hi, test_size=0.3, random_state=0)

for name, model in MODELS:
    try:
        m = model.fit(Xh_tr, yh_tr)
        auc_val = roc_auc_score(yh_te, m.predict_proba(Xh_te)[:,1])
        print(f'  {name:5s}: AUC = {auc_val:.4f}')
    except Exception as e:
        print(f'  {name:5s}: FAILED — {type(e).__name__}: {str(e)[:60]}')

In [ ]:
# visualising the boundary reveals the shape each model can express
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
x_min, x_max = X_s1[:,0].min()-1, X_s1[:,0].max()+1
y_min_p, y_max_p = X_s1[:,1].min()-1, X_s1[:,1].max()+1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                      np.linspace(y_min_p, y_max_p, 200))
for ax, (name, model) in zip(axes, MODELS):
    model.fit(X_s1, y_s1)
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap='RdBu')
    ax.scatter(X0[:,0], X0[:,1], s=10, alpha=0.7, label='Class 0')
    ax.scatter(X1[:,0], X1[:,1], s=10, alpha=0.7, color='C3', label='Class 1')
    ax.set_title(name, fontsize=12)
    ax.set_xlabel('x₁'); ax.set_ylabel('x₂') if ax == axes[0] else None
plt.suptitle('Decision boundaries — Scenario 1 (Gaussian, shared Σ)', fontsize=11)
plt.tight_layout(); plt.show()
print('LDA and LR give the same linear boundary. QDA bends slightly. NB uses axis-aligned ellipses.')

## 3. Benchmark on Real Datasets

Real benchmarks should use nested cross-validation to avoid selection bias: when you both choose the best model and report its performance on the same folds, you optimistically inflate the estimated error rate.

In [ ]:
# Benchmark on Default dataset
X_def = d[['balance', 'income']].to_numpy()
y_def = d['y'].to_numpy()

print('Default dataset (balance + income, binary):  5-fold CV AUC')
results_def = {}
# same preprocessing for all classifiers ensures fair comparison
for name, model in MODELS:
    # cross-validation gives a less noisy accuracy estimate than a single train/test split
    scores = cross_val_score(model, X_def, y_def, cv=5, scoring='roc_auc')
    results_def[name] = scores
    print(f'  {name:5s}: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# Benchmark on Iris
le = LabelEncoder().fit(iris['species'])
y_iris = le.transform(iris['species'])
X_iris = iris[['sepal_length','sepal_width','petal_length','petal_width']].to_numpy()

print('Iris dataset (4 features, 3 classes):  5-fold CV accuracy')
results_iris = {}
for name, model in MODELS:
    scores = cross_val_score(model, X_iris, y_iris, cv=5)
    results_iris[name] = scores
    print(f'  {name:5s}: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# Plot benchmark comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, results, title, ylabel in [
        (axes[0], results_def, 'Default dataset (binary, AUC)', 'AUC'),
        (axes[1], results_iris, 'Iris (3-class, accuracy)', 'Accuracy')]:
    names = list(results.keys())
    means = [results[n].mean() for n in names]
    stds  = [results[n].std() for n in names]
    bars = ax.bar(names, means, yerr=stds, capsize=5, color=['C0','C1','C2','C3'], alpha=0.8)
    ax.set_ylim(min(means)-0.05, min(1.0, max(means)+0.05))
    ax.set_title(title); ax.set_ylabel(ylabel)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, mean + 0.002, f'{mean:.3f}',
                ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# AUC is threshold-free — fairer than accuracy for imbalanced problems
from sklearn.metrics import roc_curve, auc
Xtr_d, Xte_d, ytr_d, yte_d = train_test_split(X_def, y_def, test_size=0.3,
                                                 random_state=0, stratify=y_def)
fig, ax = plt.subplots(figsize=(6, 6))
for name, model in MODELS:
    model.fit(Xtr_d, ytr_d)
    proba = model.predict_proba(Xte_d)[:, 1]
    fpr, tpr, _ = roc_curve(yte_d, proba)
    ax.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={auc(fpr, tpr):.3f})')
ax.plot([0,1], [0,1], 'k:', alpha=0.4, label='Random')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC curves — all four classifiers on Default')
ax.legend(fontsize=9); plt.show()

## 4. Decision Guide

```
Is p >> n or p ≈ n?
├── YES → Naive Bayes or L1 Logistic Regression
└── NO → Are features approximately Gaussian?
    ├── YES → Do classes have similar covariance?
    │   ├── YES → LDA
    │   └── NO → n large? QDA : LDA
    └── NO → Logistic Regression (most robust)

Is the decision boundary strongly non-linear?
└── YES → Decision Trees, Random Forests, SVM (Courses 7–9)
```

**Always validate with cross-validation.** The guide gives a starting point, not the answer.

A practical rule of thumb: start with LR as an interpretable baseline; if n is small, try LDA or NB next since they have fewer free parameters; only move to KNN or tree-based methods if there is clear evidence of non-linear boundaries.

In [ ]:
# All four classifiers on moons — none of them win (non-linear problem)
Xm, ym = make_moons(n_samples=600, noise=0.25, random_state=0)
print('make_moons — all four classifiers hit the linear wall:')
for name, model in MODELS:
    scores = cross_val_score(model, Xm, ym, cv=5, scoring='roc_auc')
    print(f'  {name:5s}: AUC = {scores.mean():.4f}')
print('\nAll ~0.85–0.92 — none can model the curved boundary.')
print('Random Forest (next week) will reach ~0.99 on this dataset.')

In [ ]:
# Summary table across all scenarios
summary = pd.DataFrame({
    'Scenario': ['Gaussian same Σ', 'Gaussian diff Σ (n=200)', 'Non-Gaussian', 'High-dim (p=500)', 'Non-linear (moons)'],
    'Best method': ['LDA', 'QDA', 'LR', 'NB / L1-LR', 'Trees / SVM'],
    'Runner-up': ['LR', 'LR', 'NB (KDE)', 'L1-LR', 'Kernelized SVM'],
    'Avoid': ['QDA', 'LDA', 'LDA/QDA', 'LDA/QDA', 'All linear classifiers'],
})
print(summary.to_string(index=False))

## Week 4 Recap

We covered six classification methods from ISLR Chapter 4:

1. **KNN** (Course 1): non-parametric, local, sensitive to k and scale
2. **Logistic Regression** (Course 2): linear log-odds, MLE, robust, no distributional assumptions
3. **LDA** (Course 3): generative Gaussian with shared Σ → linear boundary, Fisher projection
4. **QDA** (Course 3): separate Σ_k → quadratic boundary; needs large n
5. **Naive Bayes** (Course 4): conditional independence → O(pK) parameters; wins at high p
6. **Poisson regression** (Course 5): count data; log link; multiplicative effects
7. **Comparison** (Course 6): no free lunch — match the method to the data and validate with CV

**Key formulas to remember**:
- LR: log(p/(1-p)) = Xβ, fitted by MLE
- LDA: δ_k(x) = xᵀΣ⁻¹μ_k − ½μ_kᵀΣ⁻¹μ_k + log π_k
- NB: Pr(Y=k|X) ∝ π_k · ∏_j f_kj(x_j)
- Poisson: log(E[Y|X]) = Xβ, E[Y|X] = exp(Xβ)

---

# Exercises

## Exercise 1

**Task 1.** Run all four classifiers on the Default dataset with all three predictors
(`balance`, `income`, `student`). Compare 5-fold CV AUC. Does adding `student` help any method?

In [ ]:
# your code here

### Exercise 1 — Solution

In [ ]:
d['student_num'] = (d['student'] == 'Yes').astype(float)
X2 = d[['balance', 'income']].to_numpy()
X3 = d[['balance', 'income', 'student_num']].to_numpy()
y_def = d['y'].to_numpy()

print('AUC comparison: 2 features vs 3 features (+ student)')
print(f'{"Method":8} {"2 features":>14} {"3 features":>14}')
for name, model in MODELS:
    s2 = cross_val_score(model, X2, y_def, cv=5, scoring='roc_auc').mean()
    s3 = cross_val_score(model, X3, y_def, cv=5, scoring='roc_auc').mean()
    delta = s3 - s2
    sign = '+' if delta >= 0 else ''
    print(f'{name:8} {s2:.4f}         {s3:.4f}  ({sign}{delta:.4f})')

## Exercise 2

**Task 2.** Simulate data from Scenario 2 (Gaussian, different Σ_k) with **n=50** per class.
Does QDA still win over LDA? What happens at n=500?

In [ ]:
# your code here

### Exercise 2 — Solution

In [ ]:
Sigma0 = np.array([[1.0, 0.8], [0.8, 1.0]])
Sigma1 = np.array([[1.5, -0.7], [-0.7, 0.8]])

for n_per in [50, 500]:
    rng2 = np.random.default_rng(0)
    X0_n = rng2.multivariate_normal([0,0], Sigma0, n_per)
    X1_n = rng2.multivariate_normal([2,1], Sigma1, n_per)
    X_n = np.vstack([X0_n, X1_n])
    y_n = np.array([0]*n_per + [1]*n_per)
    print(f'n = {n_per} per class:')
    for name, model in [('LDA', LinearDiscriminantAnalysis()),
                         ('QDA', QuadraticDiscriminantAnalysis())]:
        scores = cross_val_score(model, X_n, y_n, cv=10, scoring='roc_auc')
        print(f'  {name}: AUC = {scores.mean():.4f} ± {scores.std():.4f}')
    print()

## Exercise 3

**Task 3.** For each classifier, plot the ROC curve on the full Iris dataset
(3-class: use `roc_auc_score` with `multi_class='ovr'`). Which method has the highest macro-AUC?

In [ ]:
# your code here

### Exercise 3 — Solution

In [ ]:
from sklearn.model_selection import cross_val_predict
print('Multi-class AUC (OvR, macro-averaged) on Iris:')
for name, model in MODELS:
    proba = cross_val_predict(model, X_iris, y_iris, cv=5, method='predict_proba')
    auc_val = roc_auc_score(y_iris, proba, multi_class='ovr', average='macro')
    print(f'  {name:5s}: macro-AUC = {auc_val:.4f}')

## Exercise 4

**Task 4 — Learning curves.** On the Default dataset (`balance` + `income`, binary target), plot learning curves for LR, LDA, GaussianNB, and KNN (k=15). Use `sklearn.model_selection.learning_curve` with `train_sizes` from 50 to 1000 in 10 steps and `cv=5`. Plot mean train and validation accuracy vs training set size for all four classifiers on one figure (2×2 subplots). Which classifier needs the most data to reach its plateau?

In [ ]:
# your code here

### Exercise 4 — Solution

In [ ]:
from sklearn.model_selection import learning_curve
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# LDA, LR, KNN are all sensitive to feature scale — NB and trees are not but we scale anyway for parity
lc_models = [
    ('LR',  make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))),
    ('LDA', make_pipeline(StandardScaler(), LinearDiscriminantAnalysis())),
    ('NB',  make_pipeline(StandardScaler(), GaussianNB())),
    ('KNN k=15', make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15))),
]

train_sizes = np.linspace(50, 1000, 10, dtype=int)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, (name, model) in zip(axes.ravel(), lc_models):
    # cross-validation gives a less noisy accuracy estimate than a single train/test split
    ts, train_sc, val_sc = learning_curve(
        model, X_def, y_def,
        train_sizes=train_sizes, cv=5, scoring='accuracy', n_jobs=-1
    )
    ax.plot(ts, train_sc.mean(axis=1), label='Train', linewidth=2)
    ax.fill_between(ts, train_sc.mean(1)-train_sc.std(1),
                        train_sc.mean(1)+train_sc.std(1), alpha=0.15)
    ax.plot(ts, val_sc.mean(axis=1), label='Validation', linewidth=2)
    ax.fill_between(ts, val_sc.mean(1)-val_sc.std(1),
                        val_sc.mean(1)+val_sc.std(1), alpha=0.15)
    ax.set_title(name); ax.set_xlabel('Training set size'); ax.set_ylabel('Accuracy')
    ax.legend(fontsize=9); ax.set_ylim(0.9, 1.0)

plt.suptitle('Learning curves — Default dataset', fontsize=12)
plt.tight_layout(); plt.show()
print('KNN typically needs the most data to reach its plateau; LDA and NB converge quickly.')

## Exercise 5

**Task 5 — Bias-variance tradeoff plot.** Simulate 100 datasets from a 2D Gaussian mixture (class 0: N([0,0], I), class 1: N([1.5, 1.5], I), 200 samples each). For each of five classifiers (LR, LDA, QDA, GaussianNB, KNN k=5), record test accuracy across all 100 simulations. Plot a box plot of test accuracies. Which classifier has the highest variance (widest box)?

In [ ]:
# your code here

### Exercise 5 — Solution

In [ ]:
from sklearn.metrics import accuracy_score

bv_models = [
    ('LR',  LogisticRegression(max_iter=2000)),
    ('LDA', LinearDiscriminantAnalysis()),
    ('QDA', QuadraticDiscriminantAnalysis()),
    ('NB',  GaussianNB()),
    ('KNN k=5', KNeighborsClassifier(n_neighbors=5)),
]

# same preprocessing for all classifiers ensures fair comparison
n_sim = 100
n_each = 200
records = {name: [] for name, _ in bv_models}

for seed in range(n_sim):
    rng_bv = np.random.default_rng(seed)
    X0_bv = rng_bv.multivariate_normal([0, 0], np.eye(2), n_each)
    X1_bv = rng_bv.multivariate_normal([1.5, 1.5], np.eye(2), n_each)
    X_bv = np.vstack([X0_bv, X1_bv])
    y_bv = np.array([0]*n_each + [1]*n_each)
    Xtr_bv, Xte_bv, ytr_bv, yte_bv = train_test_split(X_bv, y_bv, test_size=0.3,
                                                         random_state=seed)
    for name, model in bv_models:
        model.fit(Xtr_bv, ytr_bv)
        records[name].append(accuracy_score(yte_bv, model.predict(Xte_bv)))

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([records[n] for n, _ in bv_models], labels=[n for n, _ in bv_models])
ax.set_ylabel('Test accuracy'); ax.set_title('Bias-variance across 100 simulations (Gaussian mixture)')
plt.tight_layout(); plt.show()
for name, _ in bv_models:
    vals = records[name]
    print(f'{name:10s}: mean={np.mean(vals):.4f}  std={np.std(vals):.4f}')
print('\nQDA and KNN typically show higher variance (wider boxes) than LDA and NB.')

## Exercise 6

**Task 6 — Breast Cancer benchmark.** Load `sklearn.datasets.load_breast_cancer()`. Fit all five classifiers (LR, LDA, QDA, GaussianNB, KNN k=15) using 5-fold cross-validation. Report mean AUC for each. Which wins on this dataset? Does the winner match your expectation from the decision guide?

In [ ]:
# your code here

### Exercise 6 — Solution

In [ ]:
from sklearn.datasets import load_breast_cancer

bc = load_breast_cancer()
X_bc, y_bc = bc.data, bc.target

bc_models = [
    ('LR',     make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))),
    ('LDA',    make_pipeline(StandardScaler(), LinearDiscriminantAnalysis())),
    ('QDA',    make_pipeline(StandardScaler(), QuadraticDiscriminantAnalysis())),
    ('NB',     make_pipeline(StandardScaler(), GaussianNB())),
    ('KNN k=15', make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15))),
]

print('Breast Cancer benchmark — 5-fold CV AUC')
print(f'{"Classifier":12} {"Mean AUC":>10} {"Std":>8}')
# AUC is threshold-free — fairer than accuracy for imbalanced problems
# cross-validation gives a less noisy accuracy estimate than a single train/test split
for name, model in bc_models:
    scores = cross_val_score(model, X_bc, y_bc, cv=5, scoring='roc_auc')
    print(f'{name:12} {scores.mean():>10.4f} {scores.std():>8.4f}')

print('\nLDA or LR often wins on this nearly-linear-boundary dataset,')
print('consistent with the decision guide: Gaussian-ish features, moderate n, moderate p.')